# Defining Resources

**Guide for this lesson.** Do the exercise in the **starter** `cli-project/mcp_server.py`; the finished version is in `cli-project-complete/mcp_server.py`.

**Resources expose data; tools perform actions.** A resource is like a `GET` handler — the client reads it by **URI** (a `ReadResourceRequest` → the data). Motivating feature: `@document_name` **mentions** in the CLI — you need (1) a list of doc ids for autocomplete when the user types `@`, and (2) the contents of a mentioned doc to inject into the prompt. Those are two resources.

## Two kinds of resources

- **Direct** — a static URI, e.g. `docs://documents` (always the same address).
- **Templated** — a URI with parameters, e.g. `docs://documents/{doc_id}`. The SDK parses `{doc_id}` from the URI and passes it as a keyword arg to your function.

`mime_type` hints the client at the payload shape (`application/json`, `text/plain`, ...). The SDK **serializes your return value automatically** — return a `list`/`str` and it handles the rest.

## What to modify

**File:** `cli-project/mcp_server.py` — add both resources (below the tools):

```python
@mcp.resource(
    "docs://documents",
    mime_type="application/json",
)
def list_docs() -> list[str]:
    return list(docs.keys())


@mcp.resource(
    "docs://documents/{doc_id}",
    mime_type="text/plain",
)
def fetch_doc(doc_id: str) -> str:
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]
```

`list_docs` is the **direct** resource (all ids as JSON); `fetch_doc` is the **templated** one (one doc's text). Note `{doc_id}` in the URI matches the `doc_id` parameter name.

## Verify — list resources + resource templates

Imports the **complete** server and asks the `FastMCP` instance for its direct resources and its templated ones (they're reported separately). Safe import — `mcp.run()` is guarded by `if __name__ == "__main__"`.

> If you imported a *different* folder's `mcp_server` earlier in this kernel, restart the kernel first (`import mcp_server` caches the first one loaded). Switch `folder` to `"cli-project"` to check your own edits.

In [1]:
import sys
import os
from dotenv import find_dotenv

folder = "cli-project-complete"   # switch to "cli-project" to check your own edits
PROJECT = os.path.join(os.path.dirname(find_dotenv()), "building-with-the-claude-api", "07-mcp", folder)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

import mcp_server

resources = await mcp_server.mcp.list_resources()
templates = await mcp_server.mcp.list_resource_templates()

print("Direct resources:")
for r in resources:
    print("  " + str(r.uri) + "  (" + str(r.mimeType) + ")")

print("\nResource templates:")
for t in templates:
    print("  " + str(t.uriTemplate) + "  (" + str(t.mimeType) + ")")

Direct resources:
  docs://documents  (application/json)

Resource templates:
  docs://documents/{doc_id}  (text/plain)


## Exercise the logic (optional)

Same logic the resources run, against the in-memory `docs` — the id list and one doc's contents.

In [2]:
print("ids:", list(mcp_server.docs.keys()))
print()
print("deposition.md:", mcp_server.docs["deposition.md"])

ids: ['deposition.md', 'report.pdf', 'financials.docx', 'outlook.pdf', 'plan.md', 'spec.txt']

deposition.md: This deposition covers the testimony of Angela Smith, P.E.


## Testing in the inspector (optional)

`mcp dev mcp_server.py` shows **Resources** (direct) and **Resource Templates** (parameterized) as separate sections — click one to see the exact response your client will receive. (Same Node/token caveats as the server-inspector lesson; skip if it's fussy.)

## Where this leaves us

The **server** now exposes the doc list + doc contents as resources. But the **client** still can't read them — `mcp_client.py`'s `read_resource` is a `TODO`. That's the next lesson, **Accessing resources**, which also wires the `@`-mention feature in the CLI.